# 分层记忆（Hierarchical Memory Layers）

> **把智能体记忆组织成速度分层（L1/L2/L3），并根据访问情况自动升级和降级，就像 CPU 缓存层级一样。**

想想你的厨房。盐和胡椒就放在炉子旁边，你一秒钟就能拿到。不那么常用的香料放在台面上方的橱柜里，要多花几秒才够得着，但橱柜能装几十个罐子。大批量的食材放在走廊尽头的储藏室里，走过去最费时间，但它什么都能装。你绝不会把 50 个罐子都摆在台面上，也不会每次做饭都跑一趟储藏室去拿盐。

这套安排之所以有效，是因为物品会根据使用情况移动。做塔可的那一周，你会把孜然拿到台面上。一月过后，你又会把节日烘焙的香料放回储藏室。系统在自适应。

LLM 智能体面临同样的问题。上下文窗口快但小。向量数据库容量大，检索只要几毫秒。归档存储空间近乎无限，但延迟更高（响应更慢）。没有分层的话，智能体要么把所有东西塞进上下文（浪费 token），要么每查一条事实都去问同一个库（即使每一轮都需要的信息也要额外等待）。

**分层记忆（Hierarchical Memory Layers）**把存储拆成三层：

| 层 | 类比 | 速度 | 容量 |
|------|---------|-------|----------|
| **L1**（热层） | 台面上的香料 | 即时（就在上下文里） | 小（受 token 限制） |
| **L2**（温层） | 橱柜里的香料 | 快（向量检索） | 中（10 万条以上） |
| **L3**（冷层） | 储藏室的大宗货物 | 慢（归档查询） | 无限 |

升级和降级策略根据访问模式在各层之间移动记忆。智能体每一轮都要用的事实（比如用户的名字）放在 L1。相关但不常访问的事实（过去项目的细节）留在 L2。很少需要的记录（几个月前的对话）存放在 L3。

**你将在这个 notebook 里搭建：**
1. 一个带访问跟踪元数据的 `TieredMemory` 数据模型。
2. 一个拥有三层存储的 `HierarchicalMemoryManager`。
3. 级联检索（cascading retrieval，先查 L1，再查 L2，最后查 L3）。
4. 基于访问频率的自动升级和降级。
5. 一个使用全部三层的对话智能体。

**读完你会理解：**
- 如何把记忆拆成速度和成本各不相同的多层。
- 升级和降级策略如何让合适的数据待在合适的层。
- 这种方法什么时候优于扁平记忆，什么时候只是平添复杂度。

## 关键概念

- **L1 / 上下文层**：智能体的上下文窗口。访问最快（检索延迟为零），容量最小。存放智能体每一轮都需要的记忆。
- **L2 / 温层**：向量数据库（按含义而非精确关键词来查找条目的数据库）。能在大型语料上提供亚秒级的语义检索。
- **L3 / 冷层**：归档存储（SQL 数据库、文档存储或文件归档）。容量最大，检索成本最高。存放历史记录和很少访问的数据。
- **升级（promotion）**：把记忆从慢层移到快层。当访问频率超过阈值时触发。
- **降级（demotion，即淘汰 eviction）**：把记忆从快层移到慢层。由陈旧度（距上次访问的时间）或容量压力触发。
- **访问频率（access frequency）**：一条记忆在最近一段窗口内被检索的次数。这个计数驱动升级和降级决策。
- **级联检索（cascading retrieval）**：先查 L1，未命中就落到 L2，再落到 L3。这模仿了 CPU 缓存未命中（cache miss）在层级间的逐级下落。
- **嵌入（embedding）**：一串捕捉文本含义的数字。含义相近的两段文本，其嵌入在向量空间中的位置也彼此接近。
- **余弦相似度（cosine similarity）**：衡量两个嵌入相似程度的指标。取值范围从 -1（完全相反）到 1（含义相同）。

## Architecture

<p align="center">
  <img src="../../images/diagrams/13_hierarchical_memory_layers.svg" alt="分层记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TB
    Q["Query"] --> L1

    subgraph L1["L1: Context Window"]
        direction LR
        L1D["Fastest | Smallest\n~4k-16k tokens"]
    end

    L1 -->|"Cache Miss"| L2

    subgraph L2["L2: Vector DB"]
        direction LR
        L2D["Fast | Medium\n~100k+ memories"]
    end

    L2 -->|"Cache Miss"| L3

    subgraph L3["L3: Archive DB"]
        direction LR
        L3D["Slow | Largest\nUnlimited capacity"]
    end

    L2 -->|"Promote\n(high access freq)"| L1
    L1 -->|"Demote\n(stale / low freq)"| L2
    L3 -->|"Promote\n(re-accessed)"| L2
    L2 -->|"Demote\n(cold / aged)"| L3

    AC["Access Counter"] -.->|"Feeds"| L1
    AC -.->|"Feeds"| L2
    AC -.->|"Feeds"| L3

    style L1 fill:#2d5a2d,stroke:#4a9,color:#fff
    style L2 fill:#5a5a2d,stroke:#aa4,color:#fff
    style L3 fill:#5a2d2d,stroke:#a44,color:#fff
```

</details>

**数据流**：查询先打到 **L1**（上下文窗口）。未命中就级联到 **L2**（向量数据库），再到 **L3**（归档存储）。

**访问计数器（Access Counter）**跟踪所有层的命中频率。当 L2 中的一条记忆被频繁访问（超过升级阈值）时，它会上移到 L1。当 L1 中的记忆变得陈旧或 L1 达到容量上限时，它下移到 L2。L2 和 L3 之间遵循同样的逻辑。

这样就形成了一个自组织的层级结构，能随智能体信息需求的变化而调整。

## 环境准备

安装依赖并配置 API 访问。

你需要：
- 设置 `LLM_API_KEY` 环境变量（用于嵌入和对话补全）。
- 在工作目录准备一个 `.env` 文件，或在 shell 里导出该变量。

In [ ]:
%pip install -q python-dotenv numpy deepagents langchain-openai

导入所需的库并初始化 OpenAI 客户端。

In [ ]:
import os
import time
import uuid
import json
from dataclasses import dataclass, field, asdict
from typing import Optional
import numpy as np

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 实现

我们分四步搭建这套层级结构：

1. **嵌入辅助函数**：把文本转换成向量。
2. **TieredMemory 数据模型**：带访问跟踪。
3. **HierarchicalMemoryManager**：包含三层和级联检索。
4. **升级和降级策略**：在轮次之间运行。

### 第 1 步：嵌入辅助函数

我们需要一个函数，把文本转换成嵌入（embedding，即捕捉含义的一串数字）。主题相近的两条记忆会得到彼此接近的嵌入。L2 的语义检索将用到它。

In [ ]:
def get_embedding(text: str) -> list[float]:
    """用 OpenAI API 把文本转换成向量嵌入。"""
    return embed_texts([text])[0]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """计算两个向量之间的余弦相似度。"""
    a_arr = np.array(a)
    b_arr = np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))

### 第 2 步：TieredMemory 数据模型

每条记忆存储它的内容、嵌入、当前所在层和访问元数据。访问计数和时间戳驱动升级和降级决策。

In [ ]:
@dataclass
class TieredMemory:
    """单条记忆，记录所在层和访问跟踪信息。"""
    content: str
    memory_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    embedding: list[float] = field(default_factory=list)
    tier: str = "L2"            # L1、L2 或 L3
    access_count: int = 0        # 被检索的总次数
    last_accessed: float = 0.0   # 上次被检索的时间戳
    created_at: float = field(default_factory=time.time)

    def record_access(self) -> None:
        """访问计数加一，并更新时间戳。"""
        self.access_count += 1
        self.last_accessed = time.time()

    def staleness(self) -> float:
        """距上次访问的秒数。数值越大越陈旧。"""
        if self.last_accessed == 0.0:
            return time.time() - self.created_at
        return time.time() - self.last_accessed

    def summary(self) -> str:
        """简短的展示字符串。"""
        preview = self.content[:60] + ("..." if len(self.content) > 60 else "")
        return f"[{self.tier}] (hits={self.access_count}) {preview}"

### 第 3 步：HierarchicalMemoryManager

这是核心类。它管理三层存储：

- **L1**：一个 Python 字典（在内存中）。小而快。这些记忆会被注入每一次 LLM 提示词。
- **L2**：一个充当向量库的 Python 字典。生产环境中你会换成 Chroma、Pinecone 或 Qdrant。
- **L3**：一个充当归档的 Python 字典。生产环境中这会是一个 SQL 数据库或对象存储。

这里用字典是为了让 notebook 自包含。无论后端是什么，检索和层间移动的逻辑都不变。

In [ ]:
class HierarchicalMemoryManager:
    """带升级和降级的三层记忆系统。"""

    def __init__(
        self,
        l1_capacity: int = 5,
        promote_threshold: int = 3,
        demote_staleness_seconds: float = 300.0,
        similarity_threshold: float = 0.3,
    ):
        # 各层存储：memory_id -> TieredMemory
        self.l1: dict[str, TieredMemory] = {}
        self.l2: dict[str, TieredMemory] = {}
        self.l3: dict[str, TieredMemory] = {}

        # 策略参数
        self.l1_capacity = l1_capacity
        self.promote_threshold = promote_threshold      # 从 L2 升到 L1 所需的访问次数
        self.demote_staleness = demote_staleness_seconds # 闲置多少秒后从 L1 降到 L2
        self.similarity_threshold = similarity_threshold # 检索所需的最小余弦相似度

        # 事件日志，用于观察层间移动
        self.event_log: list[str] = []

    # ── 存储 ─────────────────────────────────────────────────────
    def store(self, content: str, tier: str = "L2") -> TieredMemory:
        """新增一条记忆。默认放在 L2（温层）。"""
        embedding = get_embedding(content)
        memory = TieredMemory(
            content=content,
            embedding=embedding,
            tier=tier,
        )
        self._place(memory)
        self._log(f"STORE  -> {memory.tier}: {content[:50]}")
        return memory

    def _place(self, memory: TieredMemory) -> None:
        """把记忆放入其所属层的存储。"""
        tier_store = {"L1": self.l1, "L2": self.l2, "L3": self.l3}[memory.tier]
        tier_store[memory.memory_id] = memory

`query` 方法实现级联检索。它先查 L1，如果结果不够再查 L2，然后是 L3。每次命中都会增加该记忆的访问计数，为后面的升级逻辑提供依据。

In [ ]:
# ── Retrieve ──────────────────────────────────────────────────
def _query(self, query_text: str, top_k: int = 3) -> list[TieredMemory]:
    """级联检索：先查 L1，再查 L2，最后查 L3。"""
    query_emb = get_embedding(query_text)
    results: list[TieredMemory] = []

    # 先查 L1（精确的上下文内记忆）
    l1_hits = self._search_tier(self.l1, query_emb, top_k)
    results.extend(l1_hits)

    # 结果不够就落到 L2
    if len(results) < top_k:
        remaining = top_k - len(results)
        l2_hits = self._search_tier(self.l2, query_emb, remaining)
        results.extend(l2_hits)

    # 还不够就落到 L3
    if len(results) < top_k:
        remaining = top_k - len(results)
        l3_hits = self._search_tier(self.l3, query_emb, remaining)
        results.extend(l3_hits)

    # 为每个命中记录一次访问
    for mem in results:
        mem.record_access()

    return results

def __search_tier(
    self,
    tier_store: dict[str, TieredMemory],
    query_emb: list[float],
    top_k: int,
) -> list[TieredMemory]:
    """在单层内做语义检索。"""
    if not tier_store:
        return []

    scored = []
    for mem in tier_store.values():
        sim = cosine_similarity(query_emb, mem.embedding)
        if sim >= self.similarity_threshold:
            scored.append((sim, mem))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [mem for _, mem in scored[:top_k]]

HierarchicalMemoryManager.query = _query

HierarchicalMemoryManager._search_tier = __search_tier


`_move` 辅助方法负责在层之间转移记忆。`run_maintenance` 方法执行升级/降级清扫。它检查四条规则：把访问频繁的 L2 记忆升到 L1，把陈旧的 L1 记忆降到 L2，把冷清的 L2 记忆降到 L3，把被再次访问的 L3 记忆升回 L2。在智能体轮次之间调用它，保持各层平衡。

In [ ]:
# ── Tier movement ─────────────────────────────────────────────
def __move(self, memory: TieredMemory, target_tier: str) -> None:
    """把一条记忆从当前层移到目标层。"""
    source_store = {"L1": self.l1, "L2": self.l2, "L3": self.l3}[memory.tier]
    target_store = {"L1": self.l1, "L2": self.l2, "L3": self.l3}[target_tier]

    # 从源层移除
    source_store.pop(memory.memory_id, None)

    # 更新层标签并放入目标层
    old_tier = memory.tier
    memory.tier = target_tier
    target_store[memory.memory_id] = memory

    self._log(f"MOVE   {old_tier} -> {target_tier}: {memory.content[:50]}")

# ── 维护（升级/降级清扫）──────────────────
def _run_maintenance(self) -> list[str]:
    """执行一轮升级/降级。在智能体轮次之间调用。"""
    actions: list[str] = []

    # 1. 把热点 L2 记忆升到 L1
    for mem in list(self.l2.values()):
        if mem.access_count >= self.promote_threshold:
            if len(self.l1) < self.l1_capacity:
                self._move(mem, "L1")
                actions.append(f"Promoted to L1: {mem.content[:40]}")
            else:
                # L1 已满：先淘汰最陈旧的 L1 记忆
                stalest = max(self.l1.values(), key=lambda m: m.staleness())
                self._move(stalest, "L2")
                stalest.access_count = 0  # 降级时重置计数
                actions.append(f"Demoted from L1: {stalest.content[:40]}")
                self._move(mem, "L1")
                actions.append(f"Promoted to L1: {mem.content[:40]}")

    # 2. 把陈旧的 L1 记忆降到 L2
    for mem in list(self.l1.values()):
        if mem.staleness() > self.demote_staleness:
            self._move(mem, "L2")
            actions.append(f"Demoted stale L1 to L2: {mem.content[:40]}")

    # 3. 把冷清的 L2 记忆降到 L3
    for mem in list(self.l2.values()):
        if mem.staleness() > self.demote_staleness * 3:  # 归档阈值放宽 3 倍
            self._move(mem, "L3")
            actions.append(f"Demoted cold L2 to L3: {mem.content[:40]}")

    # 4. 把被再次访问的 L3 记忆升回 L2
    for mem in list(self.l3.values()):
        if mem.access_count >= 1 and mem.staleness() < self.demote_staleness:
            self._move(mem, "L2")
            actions.append(f"Promoted L3 to L2: {mem.content[:40]}")

    return actions

HierarchicalMemoryManager._move = __move

HierarchicalMemoryManager.run_maintenance = _run_maintenance


这几个工具方法补全了管理器。`get_l1_context` 把 L1 记忆格式化成文本，供 LLM 提示词使用。`tier_counts` 和 `all_memories` 用于调试和查看层级状态。

In [ ]:
# ── 上下文构建器（用于 LLM 提示词）─────────────────────
def _get_l1_context(self) -> str:
    """把 L1 记忆格式化为文本，注入 LLM 提示词。"""
    if not self.l1:
        return "No pinned memories."
    lines = []
    for mem in self.l1.values():
        lines.append(f"- {mem.content}")
    return "\n".join(lines)

# ── 工具方法 ─────────────────────────────────────────────────
def _tier_counts(self) -> dict[str, int]:
    return {"L1": len(self.l1), "L2": len(self.l2), "L3": len(self.l3)}

def _all_memories(self) -> list[TieredMemory]:
    return list(self.l1.values()) + list(self.l2.values()) + list(self.l3.values())

def __log(self, message: str) -> None:
    self.event_log.append(message)

HierarchicalMemoryManager.get_l1_context = _get_l1_context

HierarchicalMemoryManager.tier_counts = _tier_counts

HierarchicalMemoryManager.all_memories = _all_memories

HierarchicalMemoryManager._log = __log


### 第 4 步：带分层记忆的对话智能体

现在我们把记忆管理器接入一个对话智能体。这个智能体：
1. 用 LLM 从每轮对话中提取关键事实。
2. 把提取的事实默认存入 L2。
3. 回复前查询所有层获取相关上下文。
4. 每轮之后运行维护（升级/降级）。
5. 把 L1 记忆作为「固定」（pinned）上下文注入每条提示词。

In [ ]:
class HierarchicalMemoryAgent:
    """以三层记忆为后盾的对话智能体。"""

    def __init__(
        self,
        model,
        l1_capacity: int = 5,
        promote_threshold: int = 3,
        demote_staleness_seconds: float = 300.0,
    ):
        self.model = model
        self.memory = HierarchicalMemoryManager(
            l1_capacity=l1_capacity,
            promote_threshold=promote_threshold,
            demote_staleness_seconds=demote_staleness_seconds,
        )
        self.conversation: list[dict] = []  # 供 LLM 当作上下文的最近轮次
        self.max_conversation_turns = 10     # 对话中只保留最近 N 轮

    def chat(self, user_input: str) -> str:
        """处理一轮用户输入：检索、回复、提取、维护。"""
        # 1. 从所有层检索相关记忆
        retrieved = self.memory.query(user_input, top_k=3)
        retrieved_text = self._format_retrieved(retrieved)

        # 2. 用 L1 上下文和检索到的记忆构建提示词
        l1_context = self.memory.get_l1_context()

        system_prompt = (
            "You are a helpful assistant with hierarchical memory. "
            "Use the pinned memories and retrieved context to give informed answers.\n\n"
            f"## Pinned Memories (always available)\n{l1_context}\n\n"
            f"## Retrieved Context (from search)\n{retrieved_text}\n\n"
            "Keep replies concise (2-3 sentences). "
            "If you recall something from memory, mention it."
        )

        self.conversation.append({"role": "user", "content": user_input})

        # 记忆每轮变化，系统提示词每轮都不同，所以每轮重建智能体
        agent = create_deep_agent(model=self.model, system_prompt=system_prompt)
        result = agent.invoke({
            "messages": self.conversation[-self.max_conversation_turns:],
        })
        assistant_reply = to_text(result["messages"][-1].content)
        self.conversation.append({"role": "assistant", "content": assistant_reply})

        # 3. 从本轮提取并存储新事实
        self._extract_and_store(user_input)

        # 4. 运行维护（升级/降级）
        actions = self.memory.run_maintenance()

        return assistant_reply

`_extract_and_store` 方法用一次单独的 LLM 调用，从每条用户消息里提取关键事实。每条事实默认进入 L2。随着时间推移，频繁访问的事实会在维护清扫中被升到 L1。

In [ ]:
def _extract_and_store(self, user_input: str) -> None:
    """用 LLM 从用户消息中提取关键事实。"""
    extraction_response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "Extract key facts from the user message. "
                    "Return each fact on its own line. "
                    "If there are no facts worth storing, return NONE. "
                    "Do not include opinions or small talk."
                ),
            },
            {"role": "user", "content": user_input},
        ])

    facts_text = to_text(extraction_response.content).strip()
    if facts_text.upper() == "NONE" or not facts_text:
        return

    for line in facts_text.split("\n"):
        fact = line.strip().lstrip("- ").strip()
        if fact and len(fact) > 5:
            self.memory.store(fact, tier="L2")

def _format_retrieved(self, memories: list[TieredMemory]) -> str:
    if not memories:
        return "No relevant memories found."
    lines = []
    for mem in memories:
        lines.append(f"- [{mem.tier}] {mem.content}")
    return "\n".join(lines)

HierarchicalMemoryAgent._extract_and_store = _extract_and_store

HierarchicalMemoryAgent._format_retrieved = _format_retrieved


## 运行示例

我们来看看这套层级结构的实际表现。我们会：
1. 进行多轮对话，让事实存入各层。
2. 观察智能体反复访问某些事实时触发的升级。
3. 每次交互后查看各层的分布。

### 第 1 部分：向各层预置记忆

先在不同层预置一些记忆，看看级联检索如何工作。在真实系统中，这些记忆会随时间逐渐积累。

In [ ]:
manager = HierarchicalMemoryManager(
    l1_capacity=3,
    promote_threshold=2,          # 访问 2 次后升级
    demote_staleness_seconds=60,  # 演示用的短时间窗口
)

# 把记忆存入不同层
manager.store("User's name is Alice.", tier="L1")          # 固定：始终在上下文中
manager.store("Alice is a machine learning engineer.", tier="L1")

manager.store("Alice works at Acme Corp.", tier="L2")       # 温层：需要检索
manager.store("Alice is building a recommendation system.", tier="L2")
manager.store("The project uses PyTorch and FAISS.", tier="L2")

manager.store("Alice attended MIT for undergrad.", tier="L3")  # 冷层：已归档
manager.store("Alice's first job was at a startup called DataMinds.", tier="L3")

print("Tier distribution:", manager.tier_counts())
print("\nAll memories:")
for mem in manager.all_memories():
    print(f"  {mem.summary()}")

### 第 2 部分：级联检索

观察查询如何先搜 L1，再级联到 L2 和 L3。

In [ ]:
print("=" * 60)
print("Query: 'What does Alice work on?'")
print("=" * 60)

results = manager.query("What does Alice work on?", top_k=3)
for r in results:
    print(f"  Found in {r.tier}: {r.content}")

print(f"\nQuery: 'Where did Alice go to school?'")
results = manager.query("Where did Alice go to school?", top_k=2)
for r in results:
    print(f"  Found in {r.tier}: {r.content}")

print("\nEvent log:")
for event in manager.event_log:
    print(f"  {event}")

### 第 3 部分：升级实战

现在多次访问同一条 L2 记忆，看它被升到 L1。记住，我们的 `promote_threshold` 是 2。

In [ ]:
print("Before promotion:")
print(f"  Tier counts: {manager.tier_counts()}")
print()

# 多次访问 "Alice works at Acme Corp"，触发升级
for i in range(2):
    results = manager.query("Where does Alice work?", top_k=1)
    if results:
        print(f"  Access {i+1}: '{results[0].content}' (hits={results[0].access_count}, tier={results[0].tier})")

# 运行维护以应用升级
actions = manager.run_maintenance()
print("\nMaintenance actions:")
for action in actions:
    print(f"  {action}")

print(f"\nAfter promotion:")
print(f"  Tier counts: {manager.tier_counts()}")
print("\nL1 contents:")
for mem in manager.l1.values():
    print(f"  {mem.summary()}")

### 第 4 部分：完整智能体对话

现在用完整的 `HierarchicalMemoryAgent` 进行多轮对话。智能体会提取事实、存入 L2，并把频繁访问的事实升到 L1。

In [ ]:
agent = HierarchicalMemoryAgent(
    model=llm,
    l1_capacity=3,
    promote_threshold=2,
    demote_staleness_seconds=120,
)

conversation_turns = [
    "Hi! I'm Bob and I'm a software engineer at CloudScale.",
    "I'm working on a distributed caching system using Redis.",
    "What do you know about me so far?",
    "I also have a dog named Pixel. He's a corgi.",
    "Tell me again: where do I work and what am I building?",
    "My team has 8 people and we ship weekly.",
    "Remind me what my dog's name is.",
]

for turn in conversation_turns:
    print(f"\n{'='*60}")
    print(f"User: {turn}")
    reply = agent.chat(turn)
    print(f"Agent: {reply}")
    counts = agent.memory.tier_counts()
    print(f"  [Tiers: L1={counts['L1']}, L2={counts['L2']}, L3={counts['L3']}]")

我们来看看各层的最终状态。

In [ ]:
print("Final memory state across all tiers:")
print(f"\nTotal memories: {sum(agent.memory.tier_counts().values())}")
print(f"Tier counts: {agent.memory.tier_counts()}")

for tier_name, tier_store in [("L1 (Hot)", agent.memory.l1),
                               ("L2 (Warm)", agent.memory.l2),
                               ("L3 (Cold)", agent.memory.l3)]:
    print(f"\n--- {tier_name} ---")
    if not tier_store:
        print("  (empty)")
    for mem in tier_store.values():
        print(f"  {mem.summary()}")

print("\n--- Event Log (last 10) ---")
for event in agent.memory.event_log[-10:]:
    print(f"  {event}")

### 第 5 部分：持久化

生产环境的层级结构必须能在重启后恢复。这个数据模型很容易序列化（serialize，即转换成文件格式）。下面是一个最简的 JSON 存取往返。

In [ ]:
def save_hierarchy(manager: HierarchicalMemoryManager, path: str) -> None:
    """把所有层保存到 JSON 文件。"""
    data = {
        "l1": {mid: asdict(m) for mid, m in manager.l1.items()},
        "l2": {mid: asdict(m) for mid, m in manager.l2.items()},
        "l3": {mid: asdict(m) for mid, m in manager.l3.items()},
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    total = len(manager.l1) + len(manager.l2) + len(manager.l3)
    print(f"Saved {total} memories across 3 tiers to {path}")


def load_hierarchy(path: str, **kwargs) -> HierarchicalMemoryManager:
    """从 JSON 文件恢复一个 HierarchicalMemoryManager。"""
    with open(path) as f:
        data = json.load(f)
    mgr = HierarchicalMemoryManager(**kwargs)
    for tier_key, tier_store in [("l1", mgr.l1), ("l2", mgr.l2), ("l3", mgr.l3)]:
        for mid, mdata in data[tier_key].items():
            mem = TieredMemory(**{k: v for k, v in mdata.items()})
            tier_store[mid] = mem
    total = len(mgr.l1) + len(mgr.l2) + len(mgr.l3)
    print(f"Loaded {total} memories from {path}")
    return mgr


# 存取往返测试
save_hierarchy(agent.memory, "hierarchy_snapshot.json")
restored = load_hierarchy("hierarchy_snapshot.json", l1_capacity=3, promote_threshold=2)
print(f"Restored tier counts: {restored.tier_counts()}")

清理临时文件。

In [ ]:
for f in ["hierarchy_snapshot.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 权衡

### 分层记忆适用的场景

- **长期运行的智能体**：在数周或数月里积累成千上万条记忆。层级结构让活跃事实保持快速访问，其余归档。
- **对成本敏感的部署**：把所有东西塞进上下文窗口会浪费 token。L1 只放最重要的内容，减少输入 token 数量。
- **混合延迟需求**：关键事实（用户名、当前项目）始终在上下文中，历史数据仍可通过级联搜索触达。

### 失效场景

- **短对话**（20 轮以内）很少产生足以让分层发挥作用的数据。扁平列表或滑动窗口就够用。
- **调参复杂**：升级阈值、陈旧度窗口和 L1 容量都需要校准。参数不当会让记忆在层间来回抖动（thrashing），或让陈旧数据一直占着 L1。
- **冷启动问题（cold start）**：新智能体没有访问历史。在积累足够查询之前，所有记忆都停留在 L2。
- **存储开销**：每条记忆都需要一个嵌入（用于语义检索）和访问元数据。这给每次存储操作都增加了延迟和成本。

### 生产环境注意事项

- 把内存中的字典换成真实后端：L1 元数据用 Redis 或 SQLite，L2 用 Chroma 或 Pinecone，L3 用 PostgreSQL 或 S3。
- 异步（在后台）运行维护，避免阻塞响应路径。
- 给 L3 加基于 TTL（time-to-live，存活时间）的过期机制，避免存储无限增长。
- 监控层间移动速率。L1 和 L2 之间的高频往返说明升级阈值没调好。

## 延伸阅读

- Patterson & Hennessy, [*Computer Organization and Design*](https://www.elsevier.com/books/computer-organization-and-design/patterson/978-0-12-820109-1?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)，2017。讲述缓存层级（cache hierarchy）的经典教材，本方法正是受它启发。
- Packer et al., ["MemGPT: Towards LLMs as Operating Systems"](https://arxiv.org/abs/2310.08560)，2023。提出虚拟上下文管理，在主上下文和外部存储之间显式换页。
- Zhang et al., ["A Survey on the Memory Mechanism of Large Language Model Based Agents"](https://arxiv.org/abs/2404.13501)，2024。综述了包括分层方法在内的记忆架构。
- Nuxoll & Laird, ["Extending Cognitive Architecture with Episodic Memory"](https://aaai.org/papers/01560-extending-cognitive-architecture-with-episodic-memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)，AAAI 2007。在 Soar 认知架构中探索情景记忆分层。

---

*上一篇：[12 - 工作记忆与上下文窗口](../12_working_memory_context_window/) | 下一篇：[14 - 记忆巩固](../14_memory_consolidation/)*

## 🧪 动手练习

三个小挑战，加深你的理解。每个大约需要 10-30 分钟。

### 挑战 1：自定义升级阈值
修改 `HierarchicalMemoryManager` 中 L3 升 L2 和 L2 升 L1 的访问计数阈值。分别试试激进升级（低阈值）和保守升级（高阈值）。在 30 次查询中测量每种设置下的 L1 缓存命中率。

### 挑战 2：层分布随时间的变化
存入 50 条记忆，每 10 次查询后运行一次 `run_maintenance()`。每轮维护后，记录 L1、L2、L3 中各有多少条记忆。把层分布随时间的变化画成堆叠面积图。

### 挑战 3：持久化分层存储
扩展 `save_hierarchy()` 和 `load_hierarchy()`，把每一层写入单独的文件（或 SQLite 表）。在新的 Python 会话中重新加载层级结构，验证查询仍能返回正确结果。这直接对应技术 21「跨会话记忆」中的持久化模式。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--13-hierarchical-memory-layers--hierarchical-memory-layers)